# Main pack setup and pack dependencies
## using  AstroTLPlot

In [ ]:
cd("/home/tomaslima/julia_tlima/pkg/AstroTLPlot/")
#import Pkg;
using Pkg;
Pkg.activate(".")
Pkg.precompile()
using Revise  # Opcional (veja abaixo)
using AstroTLPlot

using Dates
using FileIO
using Printf

Pkg.add("Colors")

using Colors
using LinearAlgebra
using Statistics
using DelimitedFiles

using Makie
using CairoMakie # Often needed for display in certain environments like VS Code or Pluto
using LaTeXStrings # For proper LaTeX rendering of scientific symbols

# 🚀 Simulation Initialization & Data Loading

This section orchestrates the loading of configuration files, memory allocation, and the ingestion of simulation data (HDF5) and ionization tables.

---

### 1️⃣ Configuration Setup
The pipeline begins by parsing the YAML configuration.

* **Target:** `./data/config/indatpgp.yaml`
* **Error Handling:** Uses a robust validation system to catch specific failures:
    * **Code 1:** File Not Found
    * **Code 2:** No Permissions
    * **Code 3:** YAML Parsing Error
    * **Code 4:** Wrong Extension

### 2️⃣ System Configuration & Allocation
Once parameters are loaded, the system performs geometric and memory calculations:

* **`configure`**: Calculates local/global volumes and grid increments.
* **`allocate_vars`**: Reserves memory for the `simulations_data` structures.
* **`read_ref_data_ds`**: Syncs the simulation timescale with the specific file indices found in `ref.dat`.

### 3️⃣ HDF5 Data Ingestion
The core simulation data is loaded via **HDF5**:

* **File Path:** Dynamically constructed using the start index (e.g., `all003000.h5`).
* **Validation:** If the data load fails, the process triggers a **CRITICAL ERROR** and halts to prevent processing null arrays.
* **Derived Fields:** `variables_v2!` calculates secondary physical quantities like temperature (`tem`), pressure (`pre`), and velocity squared (`vel2`).

### 4️⃣ Ionization & Chemical Properties
The final step prepares the chemical and ionization environment:

* **`count_ions`**: Determines the number of elements and ionization states.
* **`abundances!`**: Updates chemical composition based on configuration.
* **`ions_read_vf`**: Loads temperature-dependent ionization properties into the `tps` structure.

---

### 📋 Initialization Summary

| Phase | Key Function | Primary Output |
| :--- | :--- | :--- |
| **Config** | `load_simulation_config` | `config_data`, `pgp_data`, `runtime_data` |
| **Memory** | `allocate_vars` | `simulations_data` (Allocated Arrays) |
| **Data** | `readdata_hdf5!` | Loaded Physical Fields (Raw) |
| **Physics** | `variables_v2!` | Computed Fields (`tem`, `pre`, `vel2`) |
| **Chemistry**| `ions_read_vf` | Ionization Tables (`tps`) |

> [!IMPORTANT]
> **Check before running:** Ensure the `filename` for `ref.dat` and the HDF5 directory path in your YAML are accessible to avoid **Error Code 1** or **2**.

## Main Functions in `./scripts/main.jl`

### 1. **Initial Configuration**
- `load_simulation_config(file_name)` - Loads configuration from YAML file
  - Returns: tuple `(config_data, pgp_data, runtime_data, modions_data)` or error code
  - Error codes: 1 (File Not Found), 2 (No Permissions), 3 (YAML Parsing Error), 4 (Wrong Extension)

### 2. **Simulation Setup**
- `configure(config_data, pgp_data, runtime_data)` - Calculates configuration parameters
- `allocate_vars(config_data, pgp_data, runtime_data)` - Allocates memory for simulation variables
- `read_ref_data_ds(filename, nfiles, timescale)` - Reads reference data and generates `(time, files)`

### 3. **Data Loading**
- `readdata_hdf5!(datafile, simulations_data, config_data, runtime_data)` - Loads HDF5 simulation data

### 4. **Variables and Ions Processing**
- `variables!(simulations_data, config_data)` - Processes variables (density, velocity, etc.)
- `count_ions(config_data)` - Counts ions and determines chemical composition
- `abundances!(config_data, elem)` - Updates element abundances
- `ions_read(config_data, pgp_data, runtime_data, modions_data, elem)` - Loads ion properties
- `create_ionproperties()` - Initializes ion properties structure

In [ ]:
include("./scripts/main.jl") 

## Post-Processing and Visualization

The execution of `main.jl` loads and processes all simulation data, making it ready for analysis. The following cells demonstrate examples of using the implemented functions for calculations and visualization.


## Data Available for Analysis

After executing `main.jl`, all simulation data is loaded and processed, making it available for calculations and visualization through the implemented functions.

Below are some examples of how to use these functions.

## 📊 Statistical Analysis
This section extracts a specific 2D slice from the 3D simulation volume and computes its statistical properties (e.g., Mean, Median, Standard Deviation, Min/Max).
**Parameter:** `z_index = 1` (selected layer on Z-axis)

**Process:**
1. Extract matrix: `simulations_data.den[:, :, z_index]`
2. Compute statistics: `statistics_data2D()`
3. Display results: `print_statistics()`

**Goal:** Visualize and analyze statistics of the selected layer.

In [ ]:
z_index = 1
matriz_2d = simulations_data.den[:, :, z_index]

result_statistics = statistics_data(matriz_2d)
#print_statistics(result_statistics)

#[SUCCESS] File generated: statistic-den-003000-042_091025_0149.txt
# list generate by writeplot
list_of_files=["statistic-den-003000-042_091025_0149.txt"]
code = export_statistics(result_statistics; list_of_files=list_of_files, sav=false, disp=true)
#=
if code == AstroTLPlot.STATUS_SAVE_SUCCESS
    println("✅ Success: ",AstroTLPlot.error_message(code)) #error_message(code))
else
    println("❌ Failure: ", error_message(code))
end
=#

### 🖼️ Batch Variable Processing: `process_simulation_list`

This function acts as the main driver for generating visualizations. It iterates through a list of physical variables, processes their data, and exports plots following a **FAIR-compliant** directory structure.

#### 📋 Workflow
1. **ID Generation:** Converts the `filenum` (e.g., `3000`) into a canonical snapshot string (e.g., `"all003000"`).
2. **Iteration:** Loops through each variable in `variable_names` (e.g., `den`, `tem`).
3. **Delegation:** Calls `process_variable` for each item, which handles:
   * Data validation and scaling.
   * Creation of dedicated subfolders.
   * Execution of the plotting engine (`mapas!`).

#### 🛠️ Key Arguments
| Argument | Type | Description |
| :--- | :--- | :--- |
| `simdata` | `SimulationData` | Container with grids and raw simulation arrays. |
| `variable_names` | `Vector{String}` | List of variables to plot (e.g., `["den", "pre"]`). |
| `filenum` | `Int/String` | Snapshot index used for naming and folder organization. |
| `formats` | `Vector{String}` | Output file types (default: `["png"]`, supports `["pdf"]`, etc.). |
| `root` | `String` | Root directory for the FAIR output tree (default: `"output_fair"`). |
| `is_ions` | `Bool` | Toggle for ion-specific processing modes. |

> **Note:** The function returns `Nothing`. All results are saved directly to disk under the specified `root` path.

# Simulation Batch Export — Quick Guide (Notebook-Friendly)

This notebook call runs a batch export of figures for a list of variables and saves the resulting images to a structured folder tree. It is designed for quick, repeatable generation of plots in one pass.

---

## What the call does

```julia
# Execute simulation list processing for the specified variables
process_simulation_list(
    simulations_data;            # Simulation data structure
    kern = 0,                    # Kernel index (default: 0)
    ionz = 0,                    # Ionization index (default: 0)
    is_ions = false,             # Flag for ion species (default: false)
    cfg = config_data,           # ConfigData
    pgp = pgp_data,              # PGPData
    rt = runtime_data,           # RuntimeData
    modions = modions_data,      # ModionsData
    variable_names = vars,       # Array of variable names to process
    formats = formats,           # Output formats ["pdf"|"svg"|"png"] 
    filenum = filenum,           # Snapshot number for file naming
    root = root                  # Root directory for output
)
``

# Extended list (ensure names are valid in your stack)
# vars = ["den", "mach", "pok", "pre", "ram", "rok", "tem"]

# Pre-validated list (example)
vars = ["den", "pre", "tem"]


# the base structure is:
julia_tlima/pkg/AstroTLPlot/data/output/
└── output_fair
    └── snapshots
        └── all003000
            └── variables
            ├── species
            │   └── ions
            │   │  
                └── electrons/
                    ├── ele
                    |
                    └── elez            

### Folder structure produced by call function **process_simulation_list** is:
julia_tlima/pkg/AstroTLPlot/data/output/
└── output_fair
    └── snapshots
        └── all003000
            └── variables
                ├── den
                │   ├── color
                │   │   └── den_003000-042_020326_1605.pdf
                │   ├── contour
                │   │   └── den_003000-042_020326_1605.pdf
                │   ├── heat_cont
                │   │   └── den_003000-042_020326_1606.pdf
                │   ├── pdf
                │   │   └── den_003000-042_020326_1606.pdf
                │   ├── statistics
                │   │   └── den_003000-042_020326_1605.txt
                │   └── subplots
                ├── pre
                │   ├── color
                │   │   └── pre_003000-042_020326_1606.pdf
                │   ├── contour
                │   │   └── pre_003000-042_020326_1606.pdf
                │   ├── heat_cont
                │   │   └── pre_003000-042_020326_1606.pdf
                │   ├── pdf
                │   │   └── pre_003000-042_020326_1606.pdf
                │   ├── statistics
                │   │   └── pre_003000-042_020326_1606.txt
                │   └── subplots
                │       └── pre_003000-042_020326_1707.pdf
                └── tem
                    ├── color
                    │   └── tem_003000-042_020326_1606.pdf
                    ├── contour
                    │   └── tem_003000-042_020326_1606.pdf
                    ├── heat_cont
                    │   └── tem_003000-042_020326_1606.pdf
                    ├── pdf
                    │   └── tem_003000-042_020326_1606.pdf
                    ├── statistics
                    │   └── tem_003000-042_020326_1606.txt
                    └── subplots
                        └── tem_003000-042_020326_1630.pdf

In [ ]:
# vars=["den","ene", "mach","pok","pre", "ram","rok","tem","vxx", 
#    "vyy", "vzz","vel2", "vzz","vel2","bxx","byy","bzz","sentrop","ramp","rampok"] 
    
#vars = ["den","pre","tem"] 
vars = ["den"] 
formats = ["pdf"] # List of export formats (optional)

# Identifiers for the simulation snapshot and root directory
filenum   = 3000
root      = "./data/output/output_fair"

process_simulation_list(
    simulations_data;
    kern = 0,
    ionz = 0,
    is_ions = false,
    cfg = config_data,
    pgp = pgp_data,
    rt = runtime_data,
    modions = modions_data,
    variable_names = vars,
    formats = formats,
    filenum=filenum,           
    root=root
)

### Simulation Visualization: Subplot Generation (den)
This notebook automates the generation of multi-panel visualizations for the density variable (den). The primary focus is the efficient export to vector formats (PDF/SVG) using selective rasterization to minimize disk space.

## 1. Parameter Configuration and Paths
In this cell, we define the simulation identifiers and the desired output formats.
> **Note:** Including `pdf` in the `formats` list triggers internal rasterization logic to prevent bloated file sizes. Note that `Contour` elements may trigger CairoMakie compatibility warnings as they remain vector-based.

Note: Including pdf in the formats list triggers internal rasterization logic to prevent bloated file sizes, though Contour elements may trigger CairoMakie compatibility warnings.

In [ ]:

# List of export formats (optional)
formats = ["pdf"]  #Using "pdf" or "svg" enables rasterization for heavy plot elements

# Identifiers for the simulation snapshot and root directory
filenum   = 3000
id_string = build_id_string(filenum)        # e.g., "all003000"
root      = "./data/output/output_fair"


   var_name  = "den"
   # Compute the requested variable
   data_to_plot = compute_variable(var_name, simulations_data; cfg = config_data, pgp = pgp_data, rt = runtime_data, modions = modions_data)
            
# Execute subplot generation for the 'den' variable
maps_subplot!(
    data_to_plot,                   # 3D array: data
    simulations_data.X_grid,         # xgrid
    simulations_data.Y_grid,         # ygrid
    simulations_data.Z_grid,         # zgrid
    var_name;                        # var_name
    cfg = config_data,               # ConfigData
    pgp = pgp_data,                  # PGPData 
    rt  = runtime_data,              # RuntimeData
    modions = modions_data,          # ModionsData
    formats = formats,               # List of export formats pdf|sgv|png
    id_string=id_string,             #
    root=root
)

In [ ]:
# ions!(ionp,tps,elem,simulations_data,config_data,pgp_data,runtime_data,modions_data)


# Snapshot number for the simulation
filenum   = 3000
# Generate identifier string from filenum (e.g., "all003000")
id_string = build_id_string(filenum)

# FAIR output root directory
root = "./data/output/output_fair"

# Execute ion species data processing
ions!(
    ionp,                    # Ion species parameters
    tps,                     # Time steps / time points
    elem,                    # Element / species identifier
    simulations_data,        # Simulation data structure containing all variables
    config_data,             # Configuration data (ConfigData)
    pgp_data,                # Plasma grid parameters (PGPData)
    runtime_data,            # Runtime parameters and settings (RuntimeData)
    modions_data,            # Modified ion species data (ModionsData)
    id_string = id_string,   # Identifier string for file naming (e.g., "all003000")
    root = root              # Root directory for output files
)

# Ionization Species Analysis: `ions!`

The `ions!` function is the core routine for calculating and visualizing the spatial distribution of specific ion species within the simulation grid. It processes ionization fractions, applies physical scaling, and generates comprehensive multi-view visualizations (Top, Front, Side projections) for each ionization state of a given element.

## Directory Structure Output

When executed, `ions!` creates an organized output hierarchy under the FAIR snapshot directory:

---

```julia
julia_tlima/pkg/AstroTLPlot/data/output/
└── output_fair
    └── snapshots
        └── all003000
            ├── species
            │   └── ions
            │       ├── C
            │       │   ├── C00
            │       │   │   ├── color
            │       │   │   │   └── C00_003000-042_020326_1800.pdf
            │       │   │   ├── contour
            │       │   │   │   └── C00_003000-042_020326_1800.pdf
            │       │   │   ├── heat_cont
            │       │   │   │   └── C00_003000-042_020326_1800.pdf
            │       │   │   ├── pdf
            │       │   │   └── statistics
            │       │   │       └── C00_003000-042_020326_1800.txt
            │       │   │ ...
            │       │   │  
            │       │   ├── C06
            │       │   │   ├── color
            │       │   │   │   └── C06_003000-042_020326_1802.pdf
            │       │   │   ├── contour
            │       │   │   │   └── C06_003000-042_020326_1802.pdf
            │       │   │   ├── heat_cont
            │       │   │   │   └── C06_003000-042_020326_1802.pdf
            │       │   │   ├── pdf
            │       │   │   └── statistics
            │       │   │       └── C06_003000-042_020326_1802.txt
            │       │   └── subplots
            │       │       └── C_states-0to6_2026-03-02_18-02-52.pdf
            │       ├── H
            │       │   ├── H00
            │       │   │   ├── color
            │       │   │   │   └── H00_003000-042_020326_1758.pdf
            │       │   │   ├── contour
            │       │   │   │   └── H00_003000-042_020326_1758.pdf
            │       │   │   ├── heat_cont
            │       │   │   │   └── H00_003000-042_020326_1758.pdf
            │       │   │   ├── pdf
            │       │   │   └── statistics
            │       │   │       └── H00_003000-042_020326_1758.txt
            │       │   ├── H01
            │       │   │   ├── color
            │       │   │   │   └── H01_003000-042_020326_1759.pdf
            │       │   │   ├── contour
            │       │   │   │   └── H01_003000-042_020326_1759.pdf
            │       │   │   ├── heat_cont
            │       │   │   │   └── H01_003000-042_020326_1759.pdf
            │       │   │   ├── pdf
            │       │   │   └── statistics
            │       │   │       └── H01_003000-042_020326_1759.txt
            │       │   └── subplots
            │       │       └── H_states-0to1_2026-03-02_17-59-36.pdf
            │       ├── He
            │       │      
            │       └── N
            │       ...

```
## Function Signature

---

```julia
ions!(
    ionp,                    # Ion species parameters
    tps,                     # Time points array
    elem,                    # Element symbol (e.g., "H", "He", "C")
    sml,                     # Simulation data structure
    cfg,                     # Configuration data
    pgp,                     # Plasma grid parameters
    rt,                      # Runtime parameters
    modions;                 # Modified ion species data
    formats = ["png"],       # Output formats
    base_save_path = "./data/output/ions" # Root directory for output files
)

### Function: plot_element_subplot
Visualizes all ionization states (0..ee_num) of an element in a multi-subplot grid using logarithmic heatmaps with a fixed color range, producing reproducible, publication ready scientific figures.

# 🔬 Advanced Spectroscopic Mapping
## Function: `plot_element_subplot`

The `plot_element_subplot` is a high-level visualization tool designed to map the complete ionization hierarchy of a target element. It generates a standardized grid of subplots, ensuring that each ionization stage (from neutral $0$ to $Z_{max}$) is represented with consistent scaling for comparative analysis,  producing reproducible, publication-ready scientific figures.

---

### 🎨 Key Visualization Features
To achieve **publication-ready** standards, the function implements the following graphical constraints:

* **Logarithmic Intensity Scaling:** Captured via `LogNorm` to resolve features across multiple orders of magnitude in population density.
* **Fixed Dynamic Range ($v_{min}, v_{max}$):** A unified colorbar range across all subplots, preventing visual bias and allowing direct intensity comparison between different ions.
* **Automated Grid Geometry:** Dynamically calculates the $N \times M$ layout based on the atomic number ($ee\_num$) provided.
* **Spectroscopic Notation:** Automatically formats subplot titles (e.g., *Fe I, Fe II, Fe III*) using LaTeX integration.

---

### 📊 Implementation Example

### 📉 Systematic Ionization Mapping
## Function: `plot_element_subplot`

This high-level function automates the generation of multi-panel spectroscopic grids. It is 
engineered to visualize the evolution of an element across its entire ionization hierarchy 
(from neutral $0$ to $ee\_num$), utilizing **logarithmic heatmaps** to ensure that even low-density 
states are visible and comparable.

---
### 🛠️ Function Signature & API Reference

The function follows a strict type-hinted structure to ensure data integrity during large-scale plasma simulations:

```julia
# Julia Signature Reference
plot_element_subplot(
    temp_props::TemperatureProperties,  # Thermodynamic profile and plasma parameters
    sml::SimulationData,                # Core simulation results and spatial matrices
    element::Element,                   # Target chemical element for analysis
    ee_num::Int;                        # Max ionization state to display (e.g., 0 to ee_num)

    # Optional Keyword Arguments (Plot Customization)
    ncols::Int = 3,                     # Number of columns in the subplot grid
    save_dir::AbstractString = "",      # Output directory for exported figures
    formats::Vector{String} = ["pdf"],  # Export formats (e.g., "pdf", "png", "svg")
    rasterize_scale::Int = 1,           # DPI multiplier for rasterized layers (range 1-3)
    fixed_colorrange::Tuple{Float64,Float64} = (-12.0, 2.5), # Global log-scale limits [min, max]
    z_index::Int = 1,                   # Depth slice index for 3D simulation data
    colormap = cgrad(get_palette(15)),  # Color palette for ionization intensity
    xticks::AbstractVector = 0:200:1000,# Spatial X-axis ticks in simulation units
    yticks::AbstractVector = 0:200:1000 # Spatial Y-axis ticks in simulation units
)
```
### 🚀 Use Case: Analyzing Plasma Evolution

In [ ]:
save_dir="./data/output/maps/subplots"
plot_element_subplot(tps, simulations_data, elem,1; ncols = 4,save_dir = save_dir, formats = ["png","pdf"]  )                  

### Physical Modeling: Electron Density and Ionization Mapping
This notebook demonstrates the usage of two core functions for plasma simulation analysis: electron! for density computation and plot_element_subplot for advanced visualization of ionization states.
produce the output:
output_fair/snapshots/all003000/species/electrons/
├── ele
│   ├── color
│   │   └── ele_003000-042_040326_0240.pdf
│   ├── contour
│   │   └── ele_003000-042_040326_0240.pdf
│   ├── heat_cont
│   │   └── ele_003000-042_040326_0240.pdf
│   ├── pdf
│   └── statistics
│       └── ele_003000-042_040326_0240.txt
└── elez
    ├── C
    │   ├── color
    │   │   └── elez_06-003000-042_040326_0241.pdf
    │   ├── contour
    │   │   └── elez_06-003000-042_040326_0241.pdf
    │   ├── heat_cont
    │   │   └── elez_06-003000-042_040326_0241.pdf
    │   ├── pdf
    │   └── statistics
    │       └── elez_06-003000-042_040326_0241.txt
    ├── Fe
    ├── H
    ├── He
    ├── Mg
    │  ... 
    └── Si
        ├── color
        │   └── elez_14-003000-042_040326_0242.pdf
        ├── contour
        │   └── elez_14-003000-042_040326_0242.pdf
        ├── heat_cont
        │   └── elez_14-003000-042_040326_0242.pdf
        ├── pdf
        └── statistics
            └── elez_14-003000-042_040326_0242.txt





In [ ]:

filenum   = 3000
id_string = build_id_string(filenum)  # "all003000"

# 2) Raiz FAIR (como pediste)
root = "./data/output/output_fair"

electron!(
    tps,              # TemperatureProperties (será atualizado in-place)
    elem,             # Element
    simulations_data, # SimulationData
    config_data,      # ConfigData
    pgp_data,         # PGPData
    runtime_data,     # RuntimeData (controla se gera plots via rt.output_plot)
    modions_data;     # ModionsData
    
    # Parâmetros Opcionais (Keyword Arguments)
    formats = ["pdf"],                    # NEW: Formatos de saída formats = ["png","pdf","svg"]
    #save_path = "./data/output/electrons" # NEW: Pasta base para os outputs
    id_string=id_string,root=root
    )